# Dictionnaire de données — Awalé Boissons

**The Awalé Boissons Reporting Challenge — Kômian AI Engineer**
Version 1.0 — document de cadrage

| | |
|---|---|
| **Périmètre** | Starter dataset, du 1er janvier au 30 juin 2026 |
| **But** | Documenter les cinq sources, leurs champs, leur sens métier, les anomalies connues et les transformations prévues |

### Nature des données

Les données sont synthétiques, non nettoyées et volontairement imparfaites : certains champs numériques sont stockés comme texte, et certains champs ressemblant à des clés ne sont pas fiables. Le document distingue volontairement les faits disponibles des hypothèses à vérifier lors de l'audit.

### Décision visée

Allocation des prochains 15 M FCFA de budget marketing sur les deux prochains trimestres, avec la capacité de reproduire cette recommandation chaque mois en moins d'une heure, sans data engineer.


## 1. Vue d'ensemble

| Source | Rôle métier | Nature | Usage |
|---|---|---|---|
| `campaign_spend_export` | Dépenses et performances publicitaires | Exports paid media + saisie manuelle | Spend, impressions et clics disponibles |
| `media_plan` | Planification / facturation | Plan média Kômian | Budget prévu, facturé et dépenses hors plateformes |
| `pos_sales_daily` | Ventes quotidiennes | Points de vente | CA, unités, produits, communes, POS et canal |
| `whatsapp_orders` | Commandes de livraison | WhatsApp Business | Commandes, produits, zones, clients, revenu |
| `social_comments` | Voix des clients | Réseaux sociaux | Sentiment, thèmes, langue, spam, produit |


## 2. `campaign_spend_export`

**Anomalies connues :** absence de `campaign_id`, noms incohérents, dates mixtes, spend en FCFA/EUR, impressions et clics absents pour la radio et les influenceurs, et deux mois dupliqués. Le brief demande une clé naturelle définie, la déduplication et une règle FX documentée.

| Champ | Type | Définition | Transformation / contrôle |
|---|---|---|---|
| `platform` | text | Plateforme ou canal source. | Mapper vers canal harmonisé. |
| `campaign_name` | text | Nom libre de campagne. | Nettoyer et parser produit/mois/objectif si justifiable. |
| `date_start` | text | Début de campagne. | Parser les formats et typer en date. |
| `date_end` | text | Fin de campagne. | Parser les formats et typer en date. |
| `spend` | text | Dépense de la ligne. | Nettoyer et convertir en FCFA selon une règle FX documentée. |
| `impressions` | int, nullable | Impressions lorsqu'elles existent. | Conserver les NULL ; ne pas les transformer en zéro. |
| `clicks` | int, nullable | Clics lorsqu'ils existent. | Conserver les NULL ; calculer les taux seulement si le dénominateur existe. |
| `objective` | text | Objectif déclaré. | Normaliser si nécessaire tout en conservant la valeur source. |


## 3. `pos_sales_daily`

**Anomalies connues :** deux semaines consécutives manquantes à cause d'une panne, noms de magasins variables, `pos_id` instable et revenus négatifs correspondant aux retours. La période manquante doit rester manquante et non devenir zéro.

| Champ | Type | Définition | Transformation / contrôle |
|---|---|---|---|
| `sale_date` | date / text source | Date de vente. | Typer en date et contrôler la continuité. |
| `pos_id` | text | Identifiant POS. | Ne pas supposer sa stabilité ; résoudre les entités. |
| `pos_name` | text | Nom du POS. | Normaliser les variantes. |
| `commune` | text | Commune du POS. | Normaliser les libellés. |
| `channel` | text | retail, delivery ou ecommerce. | Conformer les catégories. |
| `product_sku` | text | Référence produit. | Conserver comme identifiant source ; enrichissement possible. |
| `units_sold` | numeric/int | Unités vendues. | Contrôler et agréger. |
| `revenue_fcfa` | numeric | Revenu ; négatif pour certains retours. | Séparer ventes/retours et calculer le CA net. |


## 4. `whatsapp_orders`

**Anomalies connues :** téléphones sous plusieurs formats, items en prose, montant manquant sur environ 1 commande sur 8, et `order_ref` répété. Le brief demande de quantifier les montants manquants et leur impact sur le CA livraison.

| Champ | Type | Définition | Transformation / contrôle |
|---|---|---|---|
| `order_ref` | text | Référence de commande. | Analyser les répétitions avant déduplication. |
| `received_at` | text | Date/heure de réception. | Parser date ou timestamp. |
| `customer_phone` | text, masked | Téléphone sous plusieurs formats. | Normaliser en `customer_key`. |
| `items_text` | text | Produits et quantités en texte libre. | Règles d'abord ; LLM pour le reliquat ; contrôle par montant. |
| `amount_fcfa` | numeric, nullable | Montant lorsqu'il est disponible. | Conserver les manquants et mesurer leur impact. |
| `delivery_zone` | text | Zone de livraison. | Normaliser et agréger. |
| `status` | text | Statut de commande. | Normaliser et définir les commandes valides pour KPI. |


## 5. `social_comments`

**Anomalies connues :** français, Nouchi et anglais mélangés, emojis et abréviations, environ 15 % de spam ou de contenu non pertinent, et absence de liaison directe entre un post et une campagne. Un échantillon annoté doit mesurer la qualité, avec attention particulière au sarcasme en Nouchi.

**Enrichissement IA :** langue, sentiment, thème, spam et produit sont dérivés par IA à partir du texte source.

| Champ | Type | Définition | Transformation / contrôle |
|---|---|---|---|
| `comment_id` | text | Identifiant commentaire. | Déduplication et traçabilité. |
| `platform` | text | Instagram, Facebook ou TikTok. | Normaliser. |
| `post_id` | text | Identifiant du post. | Lien vers campagne non garanti. |
| `published_at` | timestamp | Date/heure de publication. | Typer et dériver des périodes si utile. |
| `author_handle` | text | Identifiant public de l'auteur. | Conserver sans déduire son identité réelle. |
| `comment_text` | text | Texte du commentaire. | Nettoyage léger ; conserver le texte source. |
| `like_count` | int | Nombre de likes. | Contrôler les valeurs. |
| `reply_to_id` | text, nullable | ID du commentaire parent. | Permet de reconstruire les réponses quand possible. |


## 6. `media_plan`

**Anomalies connues :** FB/IG comptabilisés séparément de Meta, radio regroupée alors que les stations sont facturées séparément, dépenses radio/influenceurs/activation présentes uniquement ici, et absence de réconciliation avec `campaign_spend_export`. Le challenge exige de choisir la source autoritaire pour le spend et d'afficher l'écart.

| Champ | Type | Définition | Transformation / contrôle |
|---|---|---|---|
| `plan_id` | text | Identifiant du plan média. | Clé du plan. |
| `month` | text | Mois du plan. | Normaliser en période mensuelle. |
| `channel` | text | Canal prévu. | Mapper vers la dimension canal. |
| `planned_budget_fcfa` | numeric | Budget prévu. | Comparer au facturé et au spend retenu. |
| `invoiced_fcfa` | numeric, nullable | Montant facturé. | Comparer au prévu et au spend retenu. |
| `objective` | text | Objectif marketing. | Normaliser si nécessaire. |
| `owner` | text | Responsable. | Utile pour le runbook. |
| `notes` | text | Notes libres. | Conserver comme contexte, sans les transformer silencieusement en métriques. |


## 7. Modèle conceptuel cible

Le modèle cible sépare le staging, les transformations intermédiaires et les marts orientés décision : un modèle par source en staging, puis un travail de parsing, de résolution et de conformation en intermédiaire, pour aboutir à deux à quatre marts répondant directement à la décision.

| Table / dimension | Rôle | Source | Mesures / attributs |
|---|---|---|---|
| `dim_channel` | Référentiel des canaux | Spend + media plan | Canal canonique, mappings |
| `dim_pos` | Référentiel POS | POS sales | POS canonique, commune |
| `dim_product` | Référentiel produit / format, si créé | POS + WhatsApp | SKU, produit, format |
| `fct_marketing_spend` | Dépenses marketing retenues | Spend + media plan | Spend FCFA, impressions, clics |
| `fct_sales_daily` | Ventes normalisées | POS sales | Unités, CA brut, retours, CA net |
| `fct_whatsapp_order_lines` | Lignes WhatsApp parsées | WhatsApp | Produit, quantité, contrôle montant |
| `fct_customer_voice` | Commentaires enrichis | Social comments | Sentiment, thème, spam, langue, produit |


## 8. KPI et limites

| KPI / analyse | Source | Ce que cela dit | Limite |
|---|---|---|---|
| Spend par canal | Spend + media plan | Où le budget est enregistré comme dépensé. | Choix de source autoritaire à documenter. |
| CA / CA net / unités | POS sales | Évolution observée des ventes. | Deux semaines manquantes. |
| Commandes / demande livraison | WhatsApp | Structure de la demande et répétition client. | Montants manquants et parsing à contrôler. |
| Sentiment / thèmes | Social + IA | Voix client structurée. | Qualité du classifieur, Nouchi/sarcasme, spam. |
| Spend vs ventes dans le temps | Marketing + ventes | Association temporelle utile à la décision. | Pas d'attribution causale sans click-to-order. |

### Limite fondamentale

Le dataset ne permet pas de prouver qu'un canal publicitaire a causé une vente : il n'y a pas de click-to-order tracking. Le rapport doit donc distinguer comparaison temporelle et attribution causale.


## 9. Principes de gouvernance

- Tracer chaque chiffre important jusqu'à la source.
- Une donnée manquante n'est pas automatiquement zéro.
- Conserver les retours et montrer leur impact sur le CA net.
- Définir explicitement les règles de déduplication.
- Afficher les écarts de rapprochement des dépenses.
- Évaluer l'IA sur un échantillon annoté et documenter coût et erreurs.
- Ne jamais présenter une corrélation temporelle comme une causalité.

---

*Version 1.0 — document de cadrage. Les règles définitives (source autoritaire du spend, clés naturelles, déduplication et paramètres IA) seront arrêtées après audit réel des fichiers. Le challenge précise qu'il n'existe pas une unique bonne réponse : la qualité du raisonnement et sa documentation comptent.*
